In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

base = spark.read.table("spotify_catalog.view.vw_base_fact")
base=(
    base
    .filter(F.col("user_id").isNotNull())
    .groupBy("user_sk", "user_id", "user_name", "user_country", "subscription_type")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.sum(F.when(F.col("is_meaningful_play") == 1,  1)
              .otherwise(0))                                    .alias("meaningful_plays"),
        F.sum(F.when(F.col("play_category") == "Abandoned", 1)
              .otherwise(0))                                    .alias("abandoned"),
    )
    .withColumn("engagement_score",
        (F.col("meaningful_plays") * 2) +
         F.col("total_streams") -
        (F.col("abandoned") * 3)
    )
    .withColumn("engagement_tier",
        F.when(F.col("engagement_score") >= 20, "Platinum")
         .when(F.col("engagement_score") >= 10, "Gold")
         .when(F.col("engagement_score") >= 5,  "Silver")
         .otherwise("Bronze")
    )
)
base.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# all Gold aggregations read from the base view
base = spark.read.table("spotify_catalog.view.vw_base_fact")

# ─────────────────────────────────────────────────────────────────────────────
# KPI 1 — Genre + Subscription engagement
# Business Q: Which genres drive most engagement per plan?
# ─────────────────────────────────────────────────────────────────────────────

genre_sub = (
    base
    .filter(F.col("genre").isNotNull())
    .filter(F.col("subscription_type").isNotNull())
    .groupBy("genre", "subscription_type")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.countDistinct("user_id")                              .alias("unique_users"),
        F.round(F.avg("listen_duration"), 2)                    .alias("avg_listen_sec"),
        F.sum(F.when(F.col("is_meaningful_play")==1, 1)
              .otherwise(0))                                    .alias("meaningful_plays"),
        F.sum(F.when(F.col("play_category") == "Abandoned", 1)
              .otherwise(0))                                    .alias("abandoned_count"),
        F.round(
            F.sum(F.when(F.col("is_meaningful_play")==1, 1)
                  .otherwise(0)) / F.count("stream_id") * 100, 2
        )                                                       .alias("meaningful_play_rate_pct")
    )
    .withColumn("gold_ingest_timestamp", F.current_timestamp())
)

(genre_sub.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("spotify_catalog.gold.genre_subscription_kpi"))

print("genre_subscription_kpi done")

# ─────────────────────────────────────────────────────────────────────────────
# KPI 2 — Top tracks
# Business Q: Which tracks get most real listening time?
# ─────────────────────────────────────────────────────────────────────────────

top_tracks = (
    base
    .filter(F.col("track_name").isNotNull())
    .groupBy("track_id", "track_name", "artist_name", "genre")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.countDistinct("user_id")                              .alias("unique_listeners"),
        F.round(F.sum("listen_duration") / 3600, 3)            .alias("total_listen_hrs"),
        F.round(F.avg("listen_duration") /
                F.avg("duration_sec") * 100, 2)          .alias("avg_completion_pct"),
        F.round(
            F.sum(F.when(F.col("play_category") == "Skipped", 1)
                  .otherwise(0)) / F.count("stream_id") * 100, 2
        )                                                       .alias("skip_rate_pct")
    )
    .withColumn("gold_ingest_timestamp", F.current_timestamp())
)

(top_tracks.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("spotify_catalog.gold.top_tracks_kpi"))

print("top_tracks_kpi done")

# ─────────────────────────────────────────────────────────────────────────────
# KPI 3 — Device + Subscription
# Business Q: What device do Premium users prefer?
# ─────────────────────────────────────────────────────────────────────────────

device_sub = (
    base
    .filter(F.col("device_type").isNotNull())
    .filter(F.col("subscription_type").isNotNull())
    .groupBy("subscription_type", "device_type")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.countDistinct("user_id")                              .alias("unique_users"),
        F.round(F.avg("listen_duration"), 2)                    .alias("avg_listen_sec"),
        F.round(
            F.sum(F.when(F.col("play_category") == "Abandoned", 1)
                  .otherwise(0)) / F.count("stream_id") * 100, 2
        )                                                       .alias("abandon_rate_pct")
    )
    .withColumn("gold_ingest_timestamp", F.current_timestamp())
)

(device_sub.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("spotify_catalog.gold.device_subscription_kpi"))

print("device_subscription_kpi done")

# ─────────────────────────────────────────────────────────────────────────────
# KPI 4 — Monthly trend
# Business Q: How does engagement trend over time?
# ─────────────────────────────────────────────────────────────────────────────

monthly = (
    base
    .filter(F.col("year").isNotNull())
    .groupBy("year", "month")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.countDistinct("user_id")                              .alias("active_users"),
        F.round(F.sum("listen_duration") / 3600, 3)            .alias("total_listen_hrs"),
        F.round(F.avg("listen_duration") /
                F.avg("duration_sec") * 100, 2)          .alias("avg_completion_pct"),
        F.round(F.count("stream_id") /
                F.countDistinct("user_id"), 2)                 .alias("streams_per_user")
    )
    .withColumn("gold_ingest_timestamp", F.current_timestamp())
    .orderBy("year", "month")
)

(monthly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("spotify_catalog.gold.monthly_trend_kpi"))

print("monthly_trend_kpi done")

# ─────────────────────────────────────────────────────────────────────────────
# KPI 5 — User engagement scorecard
# Business Q: Rank users by engagement quality not just stream count
# Score = (meaningful_plays * 2) + total_streams - (abandoned * 3)
# ─────────────────────────────────────────────────────────────────────────────

user_scores = (
    base
    .filter(F.col("user_id").isNotNull())
    .groupBy("user_sk", "user_id", "user_name", "user_country", "subscription_type")
    .agg(
        F.count("stream_id")                                    .alias("total_streams"),
        F.sum(F.when(F.col("is_meaningful_play")==1,  1)
              .otherwise(0))                                    .alias("meaningful_plays"),
        F.sum(F.when(F.col("play_category") == "Abandoned", 1)
              .otherwise(0))                                    .alias("abandoned"),
    )
    .withColumn("engagement_score",
        (F.col("meaningful_plays") * 2) +
         F.col("total_streams") -
        (F.col("abandoned") * 3)
    )
    .withColumn("engagement_tier",
        F.when(F.col("engagement_score") >= 20, "Platinum")
         .when(F.col("engagement_score") >= 10, "Gold")
         .when(F.col("engagement_score") >= 5,  "Silver")
         .otherwise("Bronze")
    )
)

w = Window.partitionBy("subscription_type").orderBy(F.col("engagement_score").desc())
user_scores = (
    user_scores
    .withColumn("rank_in_tier", F.rank().over(w))
    .withColumn("gold_ingest_timestamp", F.current_timestamp())
)

(user_scores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("spotify_catalog.gold.user_engagement_kpi"))

print("user_engagement_kpi done")